In [2]:
import math_toolkit
math_toolkit.activate()

# If

Write symbolic conditionals in a compact form while still getting ordinary SymPy `Piecewise` expressions.

Do not confuse `If(...)` with Python's plain `if` statement. Python `if` chooses one branch immediately, while `If(...)` keeps every branch symbolic for later algebra.

`If(cond).Then(expr).Otherwise(default)`  
`If(cond).Then(expr).If(next_cond).Then(expr).Otherwise()`


In [4]:
abs_x = (
    If(x < 0).Then(-x)
    .Otherwise(x)
)
abs_x

⎧-x  for x < 0
⎨             
⎩x   otherwise

## Core behavior

`If` builds an ordered SymPy `Piecewise` expression. Each `If(condition).Then(expr)` pair contributes one branch, and the first true condition wins. Closing the chain with `Otherwise(...)` returns the actual `Piecewise`, so substitution, differentiation, simplification, and LaTeX rendering use SymPy's normal machinery.

`Done()` closes the chain with no default expression. `Otherwise()` with no argument delegates to the same no-default behavior.


In [3]:
abs_x = (
    If(x < 0).Then(-x)
    .Otherwise(x)
)
[abs_x.subs(x, value) for value in (-3, 0, 2)]


[3, 0, 2]

## Common patterns


**Symbolic equality and composed conditions.** Conditions passed to `If(...)` must be symbolic conditions, not Python truth tests. Relations such as `x < y`, `x <= y`, `x > y`, and `x >= y` are already symbolic, so use them directly.

Use `x @Eq@ y` for symbolic equality testing. Plain `x == y` asks Python whether two objects are structurally identical, which is not the same question as a mathematical equality inside `Piecewise`.

Use `&` and `|` for symbolic `and` and `or`. You can also write `@And@` and `@Or@` for the same Boolean composition when you want the condition to match the `@Eq@` syntax. Parenthesize each relation so Python's operator precedence stays explicit.

Negate a symbolic condition with `~condition`, or with SymPy's `Not(condition)` constructor when a named form reads better. Python `not` asks for a real truth value.


In [4]:
indicator = (
    If(((x > 0) @And@ (x < 1)) @Or@ (x @Eq@ 2)).Then(1)
    .Otherwise(0)
)
indicator


Piecewise((1, Eq(x, 2) | ((x > 0) & (x < 1))), (0, True))

In [5]:
indicator = (
    If(((x > 0) @And@ (x < 1)) @Or@ (x @Eq@ 2)).Then(1)
    .Otherwise(0)
)
[indicator.subs(x, value) for value in (Rational(1, 2), 2, 3)]


[1, 1, 0]

In [6]:
nonzero_indicator = (
    If(Not(x @Eq@ 0)).Then(1)
    .Otherwise(0)
)
[nonzero_indicator.subs(x, value) for value in (-1, 0, 2)]


[1, 0, 1]

## Options and Details

Use the fluent ending that matches how much of the case split you know.

**Default branches.** The fluent form uses `.Otherwise(default)` for the final catch-all branch. When you are writing a raw `Piecewise(...)` call, the standalone `Otherwise(default)` helper provides the same final true branch.


In [7]:
Piecewise((x**2, x < 1), Otherwise(x + 1))


Piecewise((x**2, x < 1), (x + 1, True))

**No default branch.** `Done()` closes the chain without an else branch, and `Otherwise()` with no argument delegates to `Done()`. If no condition matches, the result follows SymPy's ordinary no-match behavior.


In [8]:
sign_without_zero = (
    If(x < 0).Then(-1)
    .If(x > 0).Then(1)
    .Done()
)
sign_without_zero.subs(x, 0)


nan

## Semantics

`If` returns an ordered SymPy `Piecewise`: branch order matters, and the first true condition supplies the value. Conditions must remain symbolic until substitution or evaluation decides them. With no default branch, unmatched values follow SymPy's ordinary no-match behavior.

## Examples and Applications

### Example: define a conditional symbolic function

Because the result is a normal SymPy expression, the DSL can live inside a `NamedFunction` definition and expand later with the rest of the authored symbolic formula.

In [9]:
@NamedFunction
def PositivePart(x):
    return If(x > 0).Then(x).Otherwise(0)

PositivePart(x) @Eq@ PositivePart(x)


Eq(PositivePart(x), Piecewise((x, x > 0), (0, True)))

## Pitfalls


A plain Python `bool` usually means the condition was already decided before SymPy saw it. `If` rejects that case so the mistake stays close to the line that created it.


In [10]:
try:
    If(x == x).Then(1).Otherwise(0)
except TypeError as err:
    print(err)


If(condition) received a plain Python bool. For symbolic equality, use x @Eq@ y, not x == y. For the default branch, use .Otherwise(expr).


Python's `and`, `or`, and `not` ask for real truth values, so use symbolic Boolean operators inside conditions: `&` and `|`, or `@And@` and `@Or@` for the infix-helper style, and `~condition` or `Not(condition)` for negation. When a condition is already a SymPy relation such as `x < 0`, use it directly.


## See also

[Expression](Expression.ipynb)  
[NamedFunction](NamedFunction.ipynb)  
[Function](Function.ipynb)  
[Equations and LaTeX](../tutorials/equations_and_latex.ipynb)  
[Composing expressions](../tutorials/composing_expressions.ipynb)
